# Phase 5: Grounded Non-Clinical LLM Explanation Layer Demonstration

## Music Brain Wellbeing Intelligence System

This notebook demonstrates the complete end-to-end pipeline bridging Spotify listening logs, personalized recommendations, RAG semantic research retrieval, and grounded LLM explanation generation:

$$\text{Listening History} \rightarrow \text{User Profile} \rightarrow \text{Recommendation Engine} \rightarrow \text{RAG Retriever} \rightarrow \text{EvidencePackage} \rightarrow \text{LLM Explanation}$$

> **Scientific & Safety Boundary:** The recommendation engine deterministically decides *what* to recommend. The RAG system retrieves peer-reviewed PubMed evidence. The LLM synthesizes and explains *why* the recommendation is reasonable using only the provided context. It does **not** diagnose anxiety or claim medical treatment.

In [1]:
import os
import sys
import json
import pandas as pd

# Ensure project root is in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import modules across all 5 completed phases
from src.data.music_loader import load_music_catalog
from src.features.user_profile import build_user_music_profile
from src.recommendation.recommender import recommend_tracks
from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.ingest import IngestionPipeline
from src.rag.retriever import ResearchRetriever
from src.rag.adapter import RecommendationQueryAdapter
from src.rag.evidence import build_evidence_package
from src.explanation.schemas import ExplanationRequest, SafetyConstraints
from src.explanation.explanation_generator import ExplanationGenerator

print("All Phase 1-5 module imports successful!")

All Phase 1-5 module imports successful!


## 1. Phase 1 & Phase 2/3: User Music Profile & Track Recommendation

We load the Spotify track catalog, synthesize a user listening profile, and generate top-N personalized recommendations.

In [2]:
catalog_path = os.path.join(project_root, "data", "raw", "spotify", "tracks.csv")
catalog_df = load_music_catalog(catalog_path)

# Sample demo user history
demo_history_df = pd.DataFrame([
    {"user_id": "USR_DEMO_P5", "track_id": "TRK_001", "played_at": "2026-08-28T08:00:00Z"},
    {"user_id": "USR_DEMO_P5", "track_id": "TRK_002", "played_at": "2026-08-28T08:15:00Z"},
    {"user_id": "USR_DEMO_P5", "track_id": "TRK_003", "played_at": "2026-08-28T08:30:00Z"}
])

user_profile = build_user_music_profile(demo_history_df, catalog_df)
recs_df = recommend_tracks(user_profile, catalog_df, top_n=3)

print(f"User Profile ID: {user_profile['user_id']}")
print(f"Top Recommended Tracks ({len(recs_df)}):\n")
for _, r in recs_df.iterrows():
    print(f" - {r.get('track_name', r.get('name', 'Track'))} (Genre: {r.get('genre', 'Music')}, Final Score: {r.get('final_score', 0.0):.2f})")

User Profile ID: USR_DEMO_P5
Top Recommended Tracks (3):

 - Track 1 (Genre: Metal, Final Score: nan)
 - Track 2 (Genre: Pop, Final Score: nan)
 - Track 3 (Genre: Electronic, Final Score: nan)


## 2. Phase 4: Research-Grounded RAG Retrieval

We convert the user's acoustic recommendations into a scientific search query, execute semantic search over local ChromaDB PubMed research records, and package the output as an `EvidencePackage`.

In [3]:
jsonl_path = os.path.join(project_root, "data", "raw", "research", "music_wellbeing_research.jsonl")
chroma_dir = os.path.join(project_root, "data", "vector_store", "chroma")

vector_store = VectorStore(persist_directory=chroma_dir, collection_name="music_wellbeing_research")
embedder = EmbeddingModel()
pipeline = IngestionPipeline(embedder=embedder, vector_store=vector_store)
pipeline.run(jsonl_path)

retriever = ResearchRetriever(embedder=embedder, vector_store=vector_store)
adapter = RecommendationQueryAdapter()

query = adapter.construct_query_from_profile(user_profile, target_topic="stress recovery acoustic relaxation")
retrieved_chunks = retriever.retrieve(query, top_k=2)
evidence_package = build_evidence_package(query, retrieved_chunks)

print(f"Constructed RAG Query: '{query}'")
print(f"Retrieved {len(retrieved_chunks)} research chunks across {len(evidence_package['sources'])} distinct sources.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Constructed RAG Query: 'stress recovery acoustic relaxation moderate energy acoustic structure physiological arousal stress recovery individual differences'
Retrieved 2 research chunks across 2 distinct sources.


## 3. Phase 5: Grounded LLM Explanation Generation & Validation

We assemble an `ExplanationRequest` payload and pass it to `ExplanationGenerator` in DEMO mode.

In [4]:
recs_list = recs_df.to_dict(orient="records")
acoustic_profiles = {
    "user_acoustic_summary": user_profile.get("audio_feature_summary", {}),
    "cluster_distribution": user_profile.get("cluster_distribution", {})
}

explanation_request = ExplanationRequest(
    user_profile=user_profile,
    recommendations=recs_list,
    acoustic_profiles=acoustic_profiles,
    evidence_package=evidence_package,
    safety_constraints=SafetyConstraints()
)

generator = ExplanationGenerator(mode="DEMO")
explanation_response = generator.generate(explanation_request)

print("Validated ExplanationResponse Output:\n")
print(json.dumps(explanation_response.to_dict(), indent=2))

Validated ExplanationResponse Output:

{
  "summary": "Recommended 3 tracks aligned with your acoustic preference profile. Retrieved scientific literature provides contextual observational evidence regarding these acoustic traits.",
  "recommendation_reasons": [
    "Track 'Track 1' (ID: TRK0001) matches your preference for soothing acoustics with similarity score nan.",
    "Track 'Track 2' (ID: TRK0002) matches your preference for soothing acoustics with similarity score nan.",
    "Track 'Track 3' (ID: TRK0003) matches your preference for soothing acoustics with similarity score nan."
  ],
  "observed_user_patterns": [
    "User profile indicates average session length of 0.0 tracks.",
    "Listening history displays preference for acoustic features (data provenance: real)."
  ],
  "research_context": [
    "Research by International Journal of Psychophysiology Reviewers (2024, PMID: 38458383) investigated associations between music interventions and stress recovery.",
    "Research

## 4. Grounding Validation & Safety Audit

We inspect the validation summary, grounding score, and cited PubMed sources.

In [5]:
print(f"Grounding Validation Status: {explanation_response.is_validated}")
print(f"Grounding Score: {explanation_response.grounding_score}")
print(f"Validation Warnings ({len(explanation_response.validation_warnings)}): {explanation_response.validation_warnings}")

print("\nGrounded PubMed Source Citations:")
for src in explanation_response.sources:
    print(f" - {src.get('title')} (PMID: {src.get('pmid')}, DOI: {src.get('doi')})")

Grounding Validation Status: True
Grounding Score: 1.0
Validation Warnings (0): []

Grounded PubMed Source Citations:
 - The effects of music and auditory stimulation on autonomic arousal, cognition and attention: A systematic review (PMID: 38458383, DOI: 10.1016/j.ijpsycho.2024.108420)
 - Music listening and stress recovery in healthy individuals: A systematic review with meta-analysis of experimental studies (PMID: 35714120, DOI: 10.1371/journal.pone.0270031)
